# RSNA-BCD preprocessing

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
assert (REPO_ROOT / "pyproject.toml").exists(), "Could not locate the repository root."
os.chdir(REPO_ROOT)  
sys.path.insert(0, str(REPO_ROOT))

## Load raw data

In [ ]:
from preprocessing.rsna_bcd import RsnaBCD, csv_save_path
from utils.preprocessing import (
    CRANIAL_CAUDAL, MEDIOLATERAL_OBLIQUE, birads_assessment, breast_density, dview,
    get_value, isna_v2, laterality, yes_no_mapping,
)

ds = RsnaBCD()
ds.set_dataset_name("rsna-bcd")
ds.train_df.head()

### Step 1 — drop columns that are not used downstream

In [ ]:
ds.train_df = ds.train_df.drop(columns=["site id", "machine id", "difficult negative case"])
ds.train_df.shape

### Step 2 — keep only rows with a known, non-zero BI-RADS

In [ ]:
ds.train_df = ds.train_df[ds.train_df["birads"].notna()]
ds.train_df = ds.train_df[ds.train_df["birads"] != 0]
ds.train_df.shape

### Step 3 — tag the images sub-folder, then map raw codes to harmonized labels

In [ ]:
ds.train_df["raw folder"] = "train_images"
ds.train_df["birads"] = ds.train_df["birads"].apply(lambda x: get_value(int(x), birads_assessment))
ds.train_df["cancer"] = ds.train_df["cancer"].apply(lambda x: get_value(x, yes_no_mapping))
ds.train_df["biopsy"] = ds.train_df["biopsy"].apply(lambda x: get_value(x, yes_no_mapping))
ds.train_df["implant"] = ds.train_df["implant"].apply(lambda x: get_value(x, yes_no_mapping))
ds.train_df["invasive"] = ds.train_df["invasive"].apply(lambda x: get_value(x, yes_no_mapping))
ds.train_df["laterality"] = ds.train_df["laterality"].apply(lambda x: get_value(x, laterality))
ds.train_df["density"] = ds.train_df["density"].apply(lambda x: get_value(x, breast_density) if not isna_v2(x) else None)
ds.train_df["view"] = ds.train_df["view"].apply(lambda x: get_value(x, dview))
ds.train_df["birads"].value_counts(dropna=False)

### Step 4 — keep only the two dominant exam views (CC / MLO)

In [ ]:
ds.train_df = ds.train_df[ds.train_df["view"].isin([MEDIOLATERAL_OBLIQUE, CRANIAL_CAUDAL])].reset_index(drop=True)
ds.train_df["view"].value_counts(dropna=False)

### Step 5 — rename columns for consistency with the common schema

In [ ]:
ds.train_df = ds.train_df.rename(columns={"implant": "breast implants", "density": "breast density", "view": "exam view"})
ds.train_df.columns.tolist()

## Build one exam record

In [ ]:
row = ds.train_df.iloc[0]
exam = ds.process_row(row)
exam

## Final processed CSV

In [ ]:
import json

final_df = pd.read_csv(csv_save_path)
print(f"rows before per-exam processing: {len(ds.train_df)} | rows in final csv: {len(final_df)}")
final_df.head()

In [ ]:
sample = final_df.iloc[0]
print(json.dumps(json.loads(sample['context']), indent=2))
print(json.dumps(json.loads(sample['findings']), indent=2))